In [4]:
import numpy as np
import gymnasium as gym
import random
import time

# Create the Taxi environment
env = gym.make("Taxi-v3", render_mode="rgb_array")

# Get state and action space sizes
state_space = env.observation_space.n
action_space = env.action_space.n

# Initialize Q-table
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

Qtable = initialize_q_table(state_space, action_space)

# Greedy policy
def greedy_policy(Qtable, state):
    return np.argmax(Qtable[state])

# Epsilon-greedy policy
def epsilon_greedy_policy(Qtable, state, epsilon):
    if random.uniform(0, 1) > epsilon:
        return greedy_policy(Qtable, state)
    else:
        return env.action_space.sample()

# Training parameters
n_training_episodes = 5000
learning_rate = 0.7
gamma = 0.95
max_steps = 100

# Exploration parameters
max_epsilon = 1.0
min_epsilon = 0.1
decay_rate = 0.005

# Training function
def train(Qtable):
    for episode in range(n_training_episodes):
        state, _ = env.reset()
        done = False
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

        for step in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)
            new_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            Qtable[state][action] += learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            state = new_state
            if done:
                break
    return Qtable

Qtable = train(Qtable)

print(Qtable)

[[  0.           0.           0.           0.           0.
    0.        ]
 [  2.75197027   3.94947227   2.75142693   3.94235581   5.20997639
   -5.05066609]
 [  7.93347236   9.40287166   7.93349131   9.40343058  10.9512375
    0.39618243]
 ...
 [ -3.15755901  12.58024607  -3.26929297  -3.33055074 -11.54306177
  -11.69929637]
 [ -3.33352094   5.27446958  -3.44001192  -3.84479972  -9.76451521
  -10.17732993]
 [ 15.9461193    9.51485951  -0.973       18.           0.
    4.96999942]]


In [7]:
import imageio
import os

def evaluate_and_record(Qtable, filename="taxi_agent.gif", n_episodes=3, max_steps=100):
    frames = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        done = False

        for step in range(max_steps):
            frame = env.render()  # Capture RGB frame
            frames.append(frame)

            action = greedy_policy(Qtable, state)
            new_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state = new_state

            if done:
                frames.append(env.render())  # Add final frame
                break

    # Save frames as GIF
    imageio.mimsave(filename, frames, fps=2)
    print(f"GIF saved as: {filename}")


evaluate_and_record(Qtable=Qtable)

GIF saved as: taxi_agent.gif
